In [1]:
import polars as pl
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Polygon

In [67]:
province_counts = pl.scan_parquet(Path('../data/gold/building_counts_by_province.parquet'))
unmatched_province = pl.scan_parquet(Path('../data/gold/building_counts_by_province_unmatched.parquet'))
province_recovered = pl.scan_parquet(Path('../data/gold/building_counts_by_province_unmatched_nearest_adm1.parquet'))

district_counts = pl.scan_parquet(Path('../data/gold/building_counts_by_district.parquet'))
unmatched_district = pl.scan_parquet(Path('../data/gold/building_counts_by_district_unmatched.parquet'))
district_recovered = pl.scan_parquet(Path('../data/gold/building_counts_by_district_unmatched_nearest_adm2.parquet'))
adm2_path = Path("../data/boundaries/geoBoundaries-IDN-ADM2-districts.geojson")

out_dir = Path("../out/tables/")
out_dir.mkdir(parents=True, exist_ok=True)

In [42]:
unmatched_province_distribution = (
    province_recovered
    .group_by(['province_name', 'province_code'])
    .agg(pl.len().alias('unmatched_before_recovery'))
    .join(
        province_counts,
        on=['province_name','province_code'],
        how='left'
    )
    .with_columns([
        (pl.col('unmatched_before_recovery') / pl.col('building_count')).alias('boundary_friction_share'),
        (pl.col('unmatched_before_recovery') / pl.col('unmatched_before_recovery').sum()).alias('share_unmatched_province')
    ])
    .sort('share_unmatched_province', descending=True)
    .collect()
)

unmatched_district_distribution = (
    district_recovered
    .group_by(['district_name', 'district_code'])
    .agg(pl.len().alias('unmatched_before_recovery'))
    .join(
        district_counts,
        on=['district_name','district_code'],
        how='left'
    )
    .with_columns([
        (pl.col('unmatched_before_recovery') / pl.col('building_count')).alias('boundary_friction_share'),
        (pl.col('unmatched_before_recovery') / pl.col('unmatched_before_recovery').sum()).alias('share_unmatched_district')
    ])
    .sort('share_unmatched_district', descending=True)
    .collect()
)

In [51]:
unmatched_province_distribution.write_csv(out_dir / "province_boundary_friction.csv")
unmatched_district_distribution.write_csv(out_dir / "district_boundary_friction.csv")

In [68]:
districts = pl.from_pandas(
    gpd.read_file(adm2_path)[['shapeName', 'shapeID']]
    .rename(columns={'shapeName': 'district_name', 'shapeID': 'district_code'})[["district_name", "district_code"]]
)

missing_districts = districts.join(
    district_counts.select(['district_name', 'district_code']).collect(),
    on=['district_name', 'district_code'],
    how='anti'
).sort('district_name')

In [70]:
print(f"ADM2 districts: {districts.height}")
print(f"Observed districts: {district_counts.collect().height}")
print(f"Missing districts: {missing_districts.height}")
missing_districts

ADM2 districts: 519
Observed districts: 503
Missing districts: 16


district_name,district_code
str,str
"""Asmat""","""22746128B62198228854475"""
"""Boven Digoel""","""22746128B21273779065212"""
"""Deiyai""","""22746128B51921656212420"""
"""Jayawijaya""","""22746128B40741333574981"""
"""Lanny Jaya""","""22746128B21145180666794"""
…,…
"""Puncak""","""22746128B1222572676249"""
"""Puncak Jaya""","""22746128B50358350022934"""
"""Tolikara""","""22746128B10432975654062"""


In [73]:
province_counts.collect().write_csv(out_dir / "province_counts.csv")
district_counts.collect().write_csv(out_dir / "district_counts.csv")